# 03 — Feature EngineeringWe create new columns ("features") that give the model stronger signals than the raw columns alone.Each one is explained: what it is, and *why* it could help predict fraud.

In [ ]:
import pandas as pdimport numpy as npdf = pd.read_csv("../data/insurance_claims_cleaned.csv")print(df.shape)

## 1. Policy age at time of incident (days)**Why:** Fraud is more common on policies that are very new — someone insures something then stages a claim quickly. A short policy age is a red flag.

In [ ]:
df['incident_date'] = pd.to_datetime(df['incident_date'])df['policy_bind_date'] = pd.to_datetime(df['policy_bind_date'])df['policy_age_days'] = (df['incident_date'] - df['policy_bind_date']).dt.daysdf[['policy_bind_date','incident_date','policy_age_days']].head()

## 2. Claims per year of policy tenure**Why:** `months_as_customer` tells us how long someone has been a customer overall. Combined with how many claims they've filed (we approximate this using `total_claim_amount` presence as a claim event), a customer with many claims relative to their short tenure is unusual and worth flagging.

In [ ]:
# months_as_customer is already in years-equivalent context; convert to years for a cleaner ratiodf['customer_tenure_years'] = df['months_as_customer'] / 12# This dataset has one claim per row (no repeat-claim history available), so as a practical proxy# we compute "claim size relative to how long they've been a customer" - a very new customer with# a huge claim is more suspicious than a 10-year customer with the same claim.df['claim_amount_per_tenure_year'] = df['total_claim_amount'] / df['customer_tenure_years'].replace(0, 0.1)df[['customer_tenure_years','total_claim_amount','claim_amount_per_tenure_year']].head()

## 3. High-value claim indicator**Why:** Extremely large claims deserve extra scrutiny by nature. A simple flag (1/0) for "is this claim in the top 10% by amount" gives the model an easy, direct signal instead of relying purely on the raw number.

In [ ]:
threshold = df['total_claim_amount'].quantile(0.90)df['is_high_value_claim'] = (df['total_claim_amount'] >= threshold).astype(int)print(f"Threshold (90th percentile): {threshold:.0f}")print(df['is_high_value_claim'].value_counts())

## 4. Vehicle claim ratio (claim-amount-vs-breakdown consistency check)**Why:** The PDF asks about 'claim amount compared with repair estimate' and 'invoice variance %'. This dataset doesn't have separate repair-estimate/invoice fields, but it does break total_claim_amount into injury/property/vehicle components. An unusual split (e.g. almost the entire claim is the 'vehicle' portion with nothing for injury/property, or vice versa) can indicate a fabricated or padded claim.

In [ ]:
df['vehicle_claim_ratio'] = df['vehicle_claim'] / df['total_claim_amount']df['injury_claim_ratio'] = df['injury_claim'] / df['total_claim_amount']df['property_claim_ratio'] = df['property_claim'] / df['total_claim_amount']df[['vehicle_claim_ratio','injury_claim_ratio','property_claim_ratio']].describe()

## 5. Missing-document count**Why:** Claims missing supporting documentation (no police report, no witnesses) are inherently harder to verify and correlate with fraud risk in real insurance practice.

In [ ]:
# Count how many "supporting evidence" signals are absent/weak for this claimdf['missing_doc_count'] = (    (df['police_report_available'].isin(['NO','Unknown'])).astype(int) +    (df['witnesses'] == 0).astype(int) +    (df['property_damage'].isin(['NO','Unknown'])).astype(int))df['missing_doc_count'].value_counts().sort_index()

## 6. Incident hour bucket (odd-hour indicator)**Why:** Incidents reported at unusual hours (very late night / early morning) can correlate with lower witness availability and have shown up as a mild fraud signal in similar studies.

In [ ]:
df['is_odd_hour_incident'] = df['incident_hour_of_the_day'].apply(lambda h: 1 if (h <= 5 or h >= 22) else 0)df['is_odd_hour_incident'].value_counts()

## Summary of engineered features created

In [ ]:
new_features = [    'policy_age_days', 'customer_tenure_years', 'claim_amount_per_tenure_year',    'is_high_value_claim', 'vehicle_claim_ratio', 'injury_claim_ratio',    'property_claim_ratio', 'missing_doc_count', 'is_odd_hour_incident']print("New engineered features:")for f in new_features:    print(" -", f)print("\nDataset shape after feature engineering:", df.shape)

In [ ]:
# Save for the modelling notebookdf.to_csv("../data/insurance_claims_features.csv", index=False)print("Saved feature-engineered dataset.")